# Descargando imagenes desde Google Earth Engine

<p align="center">
  <img src="./assets/earth-engine-logo.png" alt="GEE Logo" width="200"/>
</p>

El objetivo de este archivo es presentar un tutorial de como descargar datos desde Google Earth Engine (GEE) desde su API utilizando Python. Esta forma facilita las descargas masivas de datos. Este jupyter no corresponde a un tutorial de Python, no representa un tutorial oficial del GEE y tampoco un tutorial sobre sensores satelitales.

Los codigos presentados aqui son para casos de uso basico y se recomienda customizarlos para cumplir con las necesidades de cada proyecto, para ello se puede visitar la [Documentacion GEE](https://developers.google.com/earth-engine/guides?hl=es-419) para mayor información.

Autor: Javier Norambuena Leiva

## Preparativos

Para descargar datos es necesario tener un proyecto en [Google Cloud](https://console.cloud.google.com/welcome) con Google Earth Engine habilitado.

Para ello accedemos a [crear proyecto](https://console.cloud.google.com/projectcreate), le colocamos un nombre descriptivo y despues de crearlo, activamos la [API GEE](https://console.cloud.google.com/apis/api/earthengine.googleapis.com). Esto nos dejara habilitado el servicio para descargar imagenes a traves de este proyecto.

Se recomienda, el utilizar estos bloques de codigo dentro del sistema de google e.i. Google Colab, el cual tiene un sistema integrado de Python. El bloque inferior, se lo que se realiza es que se ingresa a la cuenta de google y se solicitan los permisos necesarios, es necesario indicar el nombre del proyecto.

Otro detalle util, es revisar si el servicio de Google Drive esta activo, para que las imagenes queden guardadas en la nube.

In [ ]:
import ee
import geemap
import time
ee.Authenticate()
ee.Initialize(project=##Nombre del proyecto)

## Descargando datos radar
Para esta seccion, solo consideraremos el caso de datos de Sentinel 1, existen varios campos que se pueden considerar para filtrar imagenes. Para el caso de este tutorial solo consideraremos los siguientes:
- Fecha inicial
- Fecha final
- Polarizacion
- Orbita
- Coleccion

In [ ]:
fecha_inicial = '2026-04-01'
fecha_final = '2026-08-01'
pol = "HH"
orbit = "DESCENDING"
radar = 'COPERNICUS/S1_GRD'

In [ ]:
s1_col = (ee.ImageCollection(radar)
    .filterBounds(roi)
    .filterDate(fecha_inicial, fecha_final)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', pol))
    .filter(ee.Filter.eq('orbitProperties_pass', orbit))
    .select([pol]))

collection_filter = s1_col.toList(s1_col.size())
num_images = collection_filter.size().getInfo()
print(num_images)

El codigo anterior nos entrega el poligono

In [ ]:
for i in range(num_images):
  img = ee.Image(collection_filter.get(i))

  date_info = img.date().format('YYYY-MM-dd').getInfo()

  task = ee.batch.Export.image.toDrive(
      image=img,
      description=f'{radar}_{id_lago}_{date_info}_{pol}_{orbit}',
      folder='GEE_Downloads',
      region=roi,
      scale=10,
  )

  task.start()
  time.sleep(1)

## Descargando datos opticos

In [ ]:
cloud_cover = 50
fecha_inicial = '2014-01-01'
fecha_final = '2026-01-01'

In [ ]:
s2_col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(roi)
          .filterDate(fecha_inicial, fecha_final)
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_cover)))

collection_filter = s2_col.toList(s2_col.size())
num_images = collection_filter.size().getInfo()
print(num_images)

In [ ]:
for i in range(num_images):
  img = ee.Image(collection_filter.get(i)).select(["B2", "B3", "B4", "B8", "B11"])
  img = img.resample('bilinear')

  date_info = img.date().format('YYYY-MM-dd').getInfo()

  task = ee.batch.Export.image.toDrive(
      image=img,
      description=f'S2_{id_lago}_{date_info}',#f'S2_{id_lago}_{date_info}',
      folder='GEE_Downloads',
      region=roi,#.getInfo()['coordinates'],
      scale=10,
      crs='EPSG:4326',
      maxPixels=1e9,
  )

  task.start()
  time.sleep(0.5)
  if i % 10 == 0:
    print(f"Enviada tarea {i}/{num_images} - Fecha: {date_info}")

In [ ]:
cloud_cover = 50
fecha_inicial = '1995-01-01'
fecha_final = '2026-01-01'
landsat5_col = (ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
          .filterBounds(roi)
          .filterDate(fecha_inicial, fecha_final)
          .filter(ee.Filter.lt('CLOUD_COVER', cloud_cover)))
collection_filter = landsat5_col.toList(landsat5_col.size())
num_images = collection_filter.size().getInfo()
print(num_images)


In [ ]:
for i in range(num_images):
    img = ee.Image(collection_filter.get(i)).select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5'])
    date_info = img.date().format('YYYY-MM-dd').getInfo()

    task = ee.batch.Export.image.toDrive(
        image=img,
        description=f'L5_{id_lago}_{date_info}',
        folder='GEE_Downloads',
        region=roi,
        scale=30,  # Resolución espacial nativa de Landsat 5
        crs='EPSG:4326',
        maxPixels=1e9
    )

    task.start()
    time.sleep(0.5)  # Pausa breve para control de la API

    if i % 10 == 0:
        print(f"Enviada tarea {i}/{num_images} - Fecha: {date_info}")

In [ ]:
cloud_cover = 50
fecha_inicial = '1995-01-01'
fecha_final = '2026-01-01'
landsat7_col = (ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
          .filterBounds(roi)
          .filterDate(fecha_inicial, fecha_final)
          .filter(ee.Filter.lt('CLOUD_COVER', cloud_cover)))

collection_filter = landsat7_col.toList(landsat7_col.size())
num_images = collection_filter.size().getInfo()
print(num_images)

In [ ]:
for i in range(num_images):
    img = ee.Image(collection_filter.get(i)).select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5'])
    date_info = img.date().format('YYYY-MM-dd').getInfo()

    task = ee.batch.Export.image.toDrive(
        image=img,
        description=f'L7_{id_lago}_{date_info}',
        folder='GEE_Downloads',
        region=roi,
        scale=30,  # Resolución espacial nativa de Landsat 7
        crs='EPSG:4326',
        maxPixels=1e9
    )

    task.start()
    time.sleep(0.5)  # Pausa breve para control de la API

    if i % 20 == 0:
        print(f"Enviada tarea {i}/{num_images} - Fecha: {date_info}")

In [ ]:
landsat8_col = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
          .filterBounds(roi)
          .filterDate(fecha_inicial, fecha_final)
          .filter(ee.Filter.lt('CLOUD_COVER', cloud_cover)))

collection_filter = landsat8_col.toList(landsat8_col.size())
num_images = collection_filter.size().getInfo()
print(num_images)

In [ ]:
for i in range(num_images):
    img = ee.Image(collection_filter.get(i)).select([
      'SR_B2',
      'SR_B3',
      'SR_B4',
      'SR_B5',
      'SR_B6'
  ])
    date_info = img.date().format('YYYY-MM-dd').getInfo()

    task = ee.batch.Export.image.toDrive(
        image=img,
        description=f'L8_{id_lago}_{date_info}',
        folder='GEE_Downloads',
        region=roi,
        scale=30,  # Resolución espacial nativa de Landsat 8
        crs='EPSG:4326',
        maxPixels=1e9
    )

    task.start()
    time.sleep(0.5)  # Pausa breve para control de la API

    if i % 10 == 0:
        print(f"Enviada tarea {i}/{num_images} - Fecha: {date_info}")

## Descargando otro tipo de datos disponibles

### STRM

In [ ]:
srtm = ee.Image("USGS/SRTMGL1_003")

srtm_clip = srtm.clip(roi)

In [ ]:
task = ee.batch.Export.image.toDrive(
      image=srtm_clip,
      description=f'srtm_aysen_irj',
      folder='GEE_Downloads',
      region=roi,
      scale=30,
  )
task.start()
time.sleep(1)

## Ejemplos